# 5f — Master Unified S2D Diagnostic Architecture
## Field Drift, Physical Consistency, and Model Attractor Analysis

This master notebook integrates all three S2D diagnostic branches:

1. **Branch A: Field Drift**: Observation-relative bias $B^{\text{obs}}$, bias evolution $D^{\text{obs}}$, and direct state adjustment $J$.
2. **Branch B: Physical Consistency**: Evaporative fraction ($EF$), Bowen ratio ($BR$), soil-moisture coupling, SST contrast, energy residual.
3. **Branch C: Model Attractor**: Distances $d^{\text{obs}}(\tau)$ & $d^{\text{model}}(\tau)$, relative drift $\delta d$, **4-category movement maps**, and **2-axis regional trajectory plots** ($x = \delta d^{\text{model}}, y = \delta d^{\text{obs}}$).

### Key Principles
- Shared cohort definition (JRA55_FOSIRL vs Reanalysis, 1980–1986, May & Nov).
- Independent Daily (days 1–84) and Monthly (months 1–24) preparation.
- Multi-source inventory & readiness reporting (`daily_hindcast`, `monthly_hindcast`, `observations`, `historical`).
- Neutral S2D experiment labels: `JRA55_FOSIRL`, `Reanalysis`, `JRA55_FOSIRL − Reanalysis`.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.unified import branch_a_drift as field_drift
from workflows.diagnostics.unified import branch_b_physics as physical_consistency
from workflows.diagnostics.unified import branch_c_attractor as model_attractor
from workflows.diagnostics.unified import config as configuration
from workflows.diagnostics.unified import inventory as inventory_workflow
from workflows.diagnostics.unified import references as reference_workflow
from workflows.diagnostics.unified import run_unified as unified_workflow

from esp_lab.diagnostics.attractor_core import (
    MovementCategory, spatial_rmsd,
    compute_attractor_distances, compute_relative_movement,
    classify_movement_category, classify_spatial_movement,
    build_2axis_trajectory,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

UNIFIED_DIR = Path(configuration.__file__).resolve().parent
print('Unified S2D Pipeline imports OK')


## Dask Setup

In [ ]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local')
dask_cfg = DaskConfig(cluster_type=machine_env, workers=8, cores=4, memory='16GB')
cluster, client = get_cluster_client(dask_cfg)
print(client)


## Step 1: Multi-Source Inventory & Readiness Gate

In [ ]:
%%time
S2D_DIAG_ROOT = Path(os.environ.get('ESP_LAB_S2D_DIAG_ROOT', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag'))
FIGURE_ROOT = Path(os.environ.get('ESP_LAB_FIGURE_OUTDIR', '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag'))
OUTPUT_ROOT = S2D_DIAG_ROOT / 'multimodel' / 'unified_diagnostics'
FIGURE_OUTDIR = FIGURE_ROOT / 'unified_diagnostics'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)
inv_summary = inventory_workflow.run_inventory(output_root=OUTPUT_ROOT, verbose=True)


## Branch A: Field Drift ($B^{\text{obs}}$, $D^{\text{obs}}$, $J$)

In [ ]:
%%time
# Synthetic demo data
lat = np.linspace(-90, 90, 5)
lon = np.linspace(-180, 180, 5)
leads = list(range(1, 13))
rng = np.random.default_rng(42)

model_da = xr.DataArray(rng.random((2, 3, 12, 5, 5)) * 10 + 280, dims=['Y','M','L','lat','lon'], coords={'L': leads, 'lat': lat, 'lon': lon})
obs_da   = model_da.mean('M') - 2.0

res_a = field_drift.run_branch_a(model_da, obs_da, baseline_lead=1)
print('Branch A outputs:', list(res_a))


## Branch B: Physical Consistency ($EF$, $BR$, Coupling)

In [ ]:
%%time
lh_ref = xr.DataArray(rng.random((2, 3, 12, 5, 5)) * 50 + 20, dims=['Y','M','L','lat','lon'], coords={'L': leads, 'lat': lat, 'lon': lon})
sh_ref = xr.DataArray(rng.random((2, 3, 12, 5, 5)) * 30 + 10, dims=['Y','M','L','lat','lon'], coords={'L': leads, 'lat': lat, 'lon': lon})
lh_test = lh_ref + 5.0
sh_test = sh_ref - 2.0

res_b = physical_consistency.run_branch_b(lh_ref, sh_ref, lh_test, sh_test, configuration.MONTHLY_WINDOWS)
for win, wd in res_b.items():
    print(f'  {win:<15s} mean EF(JRA55_FOSIRL)={float(wd["ef_test"].mean()):.3f}')


## Branch C: Model Attractor (Movement Toward Model State)

Calculates:
- Distance to Obs $d^{\text{obs}}(\tau) = \| X(\tau) - O_X \|$
- Distance to E3SM Climatology $d^{\text{model}}(\tau) = \| X(\tau) - M_X \|$
- Drift changes $\delta d^{\text{obs}}(W) = d^{\text{obs}}(W) - d^{\text{obs}}(W_0)$ and $\delta d^{\text{model}}(W) = d^{\text{model}}(W) - d^{\text{model}}(W_0)$
- Initial displacements $d^{\text{obs}}(W_0)$ and $d^{\text{model}}(W_0)$
- 4-Category Movement Classification Map
- 2-Axis Regional Trajectory Curve ($x = \delta d^{\text{model}}, y = \delta d^{\text{obs}}$)


In [ ]:
%%time
e3sm_clim = model_da.mean(['Y','M']) - 1.0
res_c = model_attractor.run_branch_c(model_da, obs_da, e3sm_clim, configuration.MONTHLY_WINDOWS, baseline_lead=1)
print('Branch C outputs:', list(res_c))
if res_c['movement_category_map'] is not None:
    display(res_c['movement_category_map'])


### 2-Axis Regional Trajectory Plot ($x = \delta d^{\text{model}}, y = \delta d^{\text{obs}}$)

In [ ]:
if res_c['delta_d_obs'] is not None and res_c['delta_d_model'] is not None:
    do_ts = res_c['delta_d_obs'].mean(['lat','lon'])
    dm_ts = res_c['delta_d_model'].mean(['lat','lon'])
    pts = build_2axis_trajectory(dm_ts, do_ts, d_initial_model=1.5, d_initial_obs=2.0)
    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.axvline(0, color='k', ls='--', lw=0.8)
    xs = [p.delta_d_model for p in pts]
    ys = [p.delta_d_obs for p in pts]
    ax.plot(xs, ys, color='darkgreen', marker='o', lw=2)
    for p in pts:
        ax.annotate(p.lead_name, (p.delta_d_model, p.delta_d_obs), textcoords='offset points', xytext=(5,5), ha='left')
    ax.set_xlabel('δd^{model}  (Change in distance to E3SM climatology)', fontsize=10)
    ax.set_ylabel('δd^{obs}  (Change in distance to observations)', fontsize=10)
    ax.set_title('2-Axis Regional Trajectory: Attraction vs Improvement', fontsize=11)
    plt.tight_layout()
    plt.show()


## Shutdown

In [ ]:
close_cluster(cluster, client)
